# Lesson 27b: ANOVA in Multiple Linear Regression

ANOVA (analysis of variance) splits the variation in y into the part a model explains and the part it leaves over. A **one-way ANOVA** uses that split to compare the means of several groups. In this notebook we:

1. **Recap: four models.** Refit the hybrid-car models from Lessons 26–27.
2. **Sums of squares.** Split the variation in MSRP into SST = SSR + SSE, to see what R² measures.
3. **Class-only model.** Write one-way ANOVA as a regression on one-hot class columns (one per class, no intercept), and read its coefficients as group means.
4. **Permutation test.** Run the one-way ANOVA test: shuffle the class labels to get a p-value for H₀: all class means are equal.
5. **Class means.** Bootstrap a confidence interval for each class's mean MSRP to see which classes differ.
6. **Try it: penguins.** Run the whole one-way ANOVA workflow on a second dataset.

Section names match the **Jupyter demo** tags on the slides. Whether a model's improvement is more than chance is Lesson 30.

[Acknowledgments Page](https://ds100.org/fa23/acks/)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

rng = np.random.default_rng(3022)   # fixed seed so results match the slides

## 1. Recap: four models

Same data and features as Lesson 27: acceleration, mpg, their transformations, and one-hot encoded car class (drop-first, so Compact is the baseline).

In [ ]:
hybrid = pd.read_csv('data/hybrid.csv')
n = len(hybrid)

hybrid['acc_sq'] = hybrid['acceleration'] ** 2
hybrid['mpg_sqrt'] = np.sqrt(hybrid['mpg'])

ohe = OneHotEncoder(drop='first')
ohe.fit(hybrid[['class']])
class_names = list(ohe.get_feature_names_out())
cars = hybrid.join(pd.DataFrame(ohe.transform(hybrid[['class']]).toarray(),
                                columns=class_names, index=hybrid.index))

quant = ['acceleration', 'mpg', 'acc_sq', 'mpg_sqrt']
models = {
    'SLR':    ['acceleration'],
    'MLR V1': ['acceleration', 'mpg'],
    'MLR V2': quant,
    'MLR V3': quant + class_names,
}
n

A helper that fits a model and returns its sums of squares, R² and adjusted R².

In [ ]:
def fit_stats(data, features):
    """Fit least squares (with intercept) on data; return SST, SSR, SSE, R^2 and adjusted R^2."""
    X, y = data[features], data['msrp']
    y_hat = LinearRegression().fit(X, y).predict(X)
    sst = np.sum((y - y.mean()) ** 2)
    ssr = np.sum((y_hat - y.mean()) ** 2)
    sse = np.sum((y - y_hat) ** 2)
    r2 = ssr / sst
    p, n_rows = len(features), len(y)
    adj_r2 = 1 - (1 - r2) * (n_rows - 1) / (n_rows - p - 1)
    return {'SST': sst, 'SSR': ssr, 'SSE': sse, 'R2': r2, 'adj_R2': adj_r2}

summary = pd.DataFrame({name: fit_stats(cars, f) for name, f in models.items()}).T
summary['RMSE'] = np.sqrt(summary['SSE'] / n)
summary[['RMSE', 'R2', 'adj_R2']].round(3)

## 2. Sums of squares

This section goes with the slides on choosing a test statistic: it shows what R² measures.


For least squares with an intercept, **SST = SSR + SSE**, and **R² = SSR / SST**.

In [ ]:
ss = summary[['SST', 'SSR', 'SSE']] / 1e6          # millions of $^2
ss['SSR + SSE'] = ss['SSR'] + ss['SSE']
ss['R2'] = summary['R2']
ss.round(3)

SST is the same for every model (it doesn't depend on the features), and SSR + SSE matches it exactly. Adding features moves variation from SSE into SSR.

**Check:** R² is also Var(ŷ) / Var(y), the coefficient of determination from Lesson 24.

In [ ]:
y_hat_v2 = LinearRegression().fit(cars[quant], cars['msrp']).predict(cars[quant])
np.var(y_hat_v2) / np.var(cars['msrp']), summary.loc['MLR V2', 'R2']

## 3. Class-only model

Does mean MSRP differ across car classes? A **one-way ANOVA** compares the group means:

- H₀: every class has the same mean MSRP.
- Hₐ: at least two classes have different means.

Start by looking at the groups.

In [ ]:
order = list(cars.groupby('class')['msrp'].mean().sort_values().index)
plt.figure(figsize=(9, 4))
sns.boxplot(data=cars, x='class', y='msrp', order=order, color='lightsteelblue')
sns.stripplot(data=cars, x='class', y='msrp', order=order, size=3, color='k', alpha=0.5)
plt.axhline(cars['msrp'].mean(), ls='--', color='goldenrod', label='overall mean')
plt.legend();

In [ ]:
group_stats = cars.groupby('class')['msrp'].agg(['count', 'mean', 'std'])
group_stats.round(0)

One-way ANOVA is a regression on the one-hot class columns alone, with **one column for every class** (no `drop`) and **no intercept**.

Why no intercept? Every car has a 1 in exactly one class column, so the 7 columns add up to 1 for every car. Together they already do the intercept's job; adding an intercept too is the dummy-variable trap from Lesson 27.

In [ ]:
ohe_all = OneHotEncoder()                      # no drop: one column per class
ohe_all.fit(cars[['class']])
all_class_names = list(ohe_all.get_feature_names_out())
X_class = pd.DataFrame(ohe_all.transform(cars[['class']]).toarray(), columns=all_class_names, index=cars.index)

class_model = LinearRegression(fit_intercept=False).fit(X_class, cars['msrp'])
coefs = pd.Series(class_model.coef_, index=all_class_names)
coefs.round(0)

**Interpreting the coefficients:** each coefficient is that class's mean MSRP, so differences between coefficients are differences between class means.

Check:

In [ ]:
check = pd.DataFrame({'coefficient': coefs.to_numpy(),
                      'class mean': group_stats['mean'].to_numpy()},
                     index=group_stats.index)
check.round(0)

In [ ]:
# How much more do Large cars cost than Compact cars, on average?
coefs['class_Large'] - coefs['class_Compact']

**Same model, different encoding.** Lesson 27's drop-first encoding with an intercept gives exactly the same predictions. There, the intercept is the Compact mean and each coefficient is a difference from it:

In [ ]:
drop_first_model = LinearRegression().fit(cars[class_names], cars['msrp'])
print('drop-first intercept:', round(drop_first_model.intercept_))
print('drop-first Large coefficient:', round(drop_first_model.coef_[class_names.index('class_Large')]))
print('same predictions:', np.allclose(drop_first_model.predict(cars[class_names]), class_model.predict(X_class)))

Since each prediction is just that car's class mean, the sums of squares have special names:

- **SSR = between-group variation**: how far the class means are from the overall mean, weighted by class size.
- **SSE = within-group variation**: how far each car is from its own class mean.

SST = SSR + SSE still holds without an intercept term, because the one-hot columns add up to 1.

In [ ]:
y = cars['msrp']
y_hat = class_model.predict(X_class)                               # each car's class mean
class_mean = cars.groupby('class')['msrp'].transform('mean')

between = np.sum((class_mean - y.mean()) ** 2)
within = np.sum((y - class_mean) ** 2)
ssr = np.sum((y_hat - y.mean()) ** 2)
sse = np.sum((y - y_hat) ** 2)
sst = np.sum((y - y.mean()) ** 2)

pd.DataFrame({'between / within (by hand)': [between / 1e6, within / 1e6, between / (between + within)],
              'SSR / SSE (regression)': [ssr / 1e6, sse / 1e6, ssr / sst]},
             index=['SSR (millions of $^2)', 'SSE (millions of $^2)', 'R2']).round(3)

Class alone explains about 39% of the variance in MSRP.

## 4. Permutation test

One-way ANOVA tests H₀: all class means are equal. We run it as a permutation test, like the ones from the hypothesis-testing lessons:

1. **Shuffle the class labels.** This breaks any link between class and MSRP, so H₀ is true for the shuffled data.
2. **Recompute the test statistic.** We use the class-only model's R²: the between-group share of the variation, SSR / SST.
3. **Repeat** 2,000 times to see what R² looks like when class doesn't matter.
4. **p-value** = the fraction of shuffled R² values at least as large as the observed one.

In [ ]:
def r2_by_class(labels, y):
    """R^2 of the class-only model = between-group SS / total SS (each prediction is the class mean)."""
    group_means = y.groupby(labels).transform('mean')
    return np.sum((group_means - y.mean()) ** 2) / np.sum((y - y.mean()) ** 2)

y = cars['msrp']
r2_obs = r2_by_class(cars['class'], y)

shuffled_r2 = np.array([r2_by_class(pd.Series(rng.permutation(cars['class'].to_numpy()), index=cars.index), y)
                        for _ in range(2000)])

print('observed R^2:', round(r2_obs, 3))
print('largest shuffled R^2:', round(shuffled_r2.max(), 3))
print('p-value:', np.mean(shuffled_r2 >= r2_obs))

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.hist(shuffled_r2, bins=40, edgecolor='white', label='R² with shuffled labels')
plt.axvline(r2_obs, color='goldenrod', lw=3, label=f'observed R² = {r2_obs:.2f}')
plt.xlabel('R² of the class-only model'); plt.legend();

None of the 2,000 shuffles comes close to the observed R², so the p-value is less than 1/2,000: very strong evidence that the class means are not all equal.

Notes:

- Shuffling labels doesn't assume every class has the same spread or that the noise is normal, which matters here (class SDs range from about \$800 to \$25,000).
- Even with shuffled labels, R² is a bit above 0 (about 6/152 ≈ 0.04 on average). The permutation test accounts for that automatically, because we compare against shuffled R² values, not against 0.
- Using R² (the between-group share) as the statistic gives the same test as the classical F-statistic: for a fixed number of groups and cars, a bigger R² always means a bigger F.
- A small p-value says the means are not all equal. It doesn't say **which** classes differ; that's the next section.

## 5. Class means

The test says the class means are not all equal. Which classes differ, and how precisely do we know each mean? Bootstrap the cars as in Lesson 25: resample the rows with replacement, average MSRP within each class, and repeat. We just use `groupby`: refitting the class-only model on each resample would give the same numbers, since its coefficients are the class means.

With only 4 minivans, an occasional resample has no minivan at all. We skip those resamples.

In [ ]:
def bootstrap_class_means(data, B=2000):
    results = []
    while len(results) < B:
        resample = data.sample(len(data), replace=True, random_state=rng)
        if resample['class'].nunique() < data['class'].nunique():
            continue                                   # a class is missing: skip this resample
        results.append(resample.groupby('class')['msrp'].mean()[order].to_numpy())
    return np.array(results)

boot_means = bootstrap_class_means(cars)
lo, hi = np.percentile(boot_means, [2.5, 97.5], axis=0)
class_ci = pd.DataFrame({'n': group_stats.loc[order, 'count'],
                         'mean': group_stats.loc[order, 'mean'],
                         'CI low': lo, 'CI high': hi}).round(0)
class_ci

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.errorbar(class_ci['mean'], range(len(order)),
             xerr=[class_ci['mean'] - class_ci['CI low'], class_ci['CI high'] - class_ci['mean']],
             fmt='o', capsize=4)
plt.yticks(range(len(order)), order)
plt.axvline(cars['msrp'].mean(), ls='--', color='goldenrod', label='overall mean')
plt.xlabel('mean MSRP with bootstrap 95% CI'); plt.legend();

Two Seaters are clearly cheaper and Large cars clearly pricier than the rest: their intervals are far from the others.

Cautions:

- Non-overlapping intervals point to a difference; overlapping intervals don't prove two means are equal.
- 7 classes give 21 pairs to compare, so some pair could look different by chance.
- Small classes (4 Minivans, 6 Pickup Trucks) give wide, less reliable intervals.
- This compares classes without accounting for acceleration and mpg. Lesson 30 asks whether class still matters once we do.

### Why not just eyeball the intervals?

Why run a test at all, instead of checking whether any of the 7 intervals fail to overlap? Let's check how often eyeballing gives a **false alarm**.

Shuffle the class labels, so H₀ is true by construction (class has nothing to do with price). Then build all 7 bootstrap intervals and check whether any pair fails to overlap. Repeat 300 times.

(This uses `np.bincount` to compute the 7 class means quickly; it does the same thing as `groupby('class').mean()`, just much faster.)

In [ ]:
codes, class_labels = pd.factorize(cars['class'])          # class as integer codes 0..6
prices = cars['msrp'].to_numpy()
n_cars, n_classes = len(prices), len(class_labels)

def any_pair_separated(labels, B=1000):
    """Bootstrap 95% CIs for every class mean; True if some pair of intervals doesn't overlap."""
    means = np.empty((B, n_classes))
    for b in range(B):
        i = rng.integers(0, n_cars, n_cars)                        # resample the cars
        counts = np.bincount(labels[i], minlength=n_classes)
        with np.errstate(invalid='ignore', divide='ignore'):       # a class missing from a resample -> NaN
            means[b] = np.bincount(labels[i], weights=prices[i], minlength=n_classes) / counts
    lo, hi = np.nanpercentile(means, [2.5, 97.5], axis=0)
    return (lo[:, None] > hi[None, :]).any()                       # some interval entirely above another

false_alarms = np.mean([any_pair_separated(rng.permutation(codes)) for _ in range(300)])
print(f'H0 true, yet some pair of intervals does not overlap in {false_alarms:.0%} of simulations')

Even though class has nothing to do with price in these simulations, eyeballing "finds" a difference roughly 1 time in 3. Two things cause this:

- **Many comparisons:** 7 intervals give 21 pairs, and every extra pair is another chance for a fluke.
- **Small groups:** with only 4 minivans or 6 pickup trucks, the bootstrap has very little to resample, so those intervals can come out too narrow.

A permutation test at the 5% level, by design, raises a false alarm only about 5% of the time. That's why we **test first** with one statistic (R²), and then use the intervals to describe **which** classes differ.

### Going further (optional): comparing every pair

The test said the class means aren't all equal. Which **pairs** differ? Comparing all 21 pairs needs an adjustment for the many comparisons, or flukes add up (as we just saw). Tukey's method is the classic fix. Its idea:

> Compare each pair's gap to the **largest** gap among the 7 classes that chance alone would produce.

We can do that by shuffling, just like the permutation test:

1. For each pair, compute a standardized difference in means (difference ÷ its standard error, like a two-sample t-statistic).
2. Shuffle the class labels (H₀ true) and record the **largest** standardized difference over all 21 pairs. Repeat 2,000 times.
3. A real pair differs if its standardized difference beats the 95th percentile of those largest values. Its adjusted p-value = the fraction of shuffles whose largest value is at least as big.

Because every pair is compared to the largest chance gap, the chance of **any** false alarm across all 21 pairs stays near 5%.

In [ ]:
from itertools import combinations

class_codes = pd.Categorical(cars['class'], categories=order).codes   # classes in order of mean MSRP
pairs = list(combinations(range(len(order)), 2))                       # all 21 pairs (i, j)
pair_i, pair_j = np.array(pairs).T
n_per_class = np.bincount(class_codes, minlength=len(order))

def pairwise_t(labels):
    """Standardized difference in means (j minus i) for every pair of classes."""
    means = np.bincount(labels, weights=prices, minlength=len(order)) / n_per_class
    variances = (np.bincount(labels, weights=prices**2, minlength=len(order)) - n_per_class * means**2) / (n_per_class - 1)
    se = np.sqrt(variances[pair_i] / n_per_class[pair_i] + variances[pair_j] / n_per_class[pair_j])
    return (means[pair_j] - means[pair_i]) / se

t_obs = pairwise_t(class_codes)
max_shuffled = np.array([np.abs(pairwise_t(rng.permutation(class_codes))).max() for _ in range(2000)])

class_means = group_stats.loc[order, 'mean'].to_numpy()
pair_results = pd.DataFrame({
    'pair': [f'{order[j]} vs {order[i]}' for i, j in pairs],
    'difference ($)': class_means[pair_j] - class_means[pair_i],
    'standardized': np.abs(t_obs),
    'adjusted p': [np.mean(max_shuffled >= abs(t)) for t in t_obs],
})
pair_results['differs'] = pair_results['adjusted p'] < 0.05
print('cutoff (95th percentile of the largest shuffled value):', round(np.percentile(max_shuffled, 95), 2))
pair_results.sort_values('adjusted p').round(3)

10 of the 21 pairs differ: Two Seaters are cheaper than every class except Minivans (only 4 cars), Large cars are pricier than every class except SUVs, and SUVs are pricier than Compacts.

For comparison, classical Tukey's method (`scipy.stats.tukey_hsd`) assumes every class has the **same spread**, which is far from true here (SDs from about \$800 to \$25,000). It pools the big spreads of the Large and Midsize classes into every comparison, so it misses Two Seater vs Compact:

In [ ]:
from scipy import stats
tukey = stats.tukey_hsd(*[cars.loc[cars['class'] == c, 'msrp'] for c in order])
print('classical Tukey flags', int((tukey.pvalue[np.triu_indices(len(order), 1)] < 0.05).sum()), 'of 21 pairs')
print('Two Seater vs Compact, classical Tukey p =', round(tukey.pvalue[0, 1], 3))

---

## 6. Try it: penguins

Researchers at Palmer Station, Antarctica, measured penguins of three species. These birds are a **sample**; the question is about the **population** of all Adelie, Chinstrap and Gentoo penguins:

> Do the three species have different mean body mass?

Run the same steps as for the cars. (Solutions are below each prompt.)

In [ ]:
penguins = sns.load_dataset('penguins').dropna(subset=['body_mass_g'])
penguins.groupby('species')['body_mass_g'].agg(['count', 'mean', 'std']).round(1)

**Step 1.** State H₀ and Hₐ in terms of the population.

*Solution:* H₀: the three species have the same population mean body mass. Hₐ: at least one species' population mean is different.

**Step 2.** Make a box plot of body mass by species.

In [ ]:
sns.boxplot(data=penguins, x='species', y='body_mass_g', color='lightsteelblue')
sns.stripplot(data=penguins, x='species', y='body_mass_g', size=3, color='k', alpha=0.4);

**Step 3.** Fit the class-only model: one-hot encode `species` (all 3 columns) and fit with no intercept. Check that the coefficients are the species means. What is R²?

In [ ]:
ohe_species = OneHotEncoder()
X_species = pd.DataFrame(ohe_species.fit_transform(penguins[['species']]).toarray(),
                         columns=ohe_species.get_feature_names_out(), index=penguins.index)
species_model = LinearRegression(fit_intercept=False).fit(X_species, penguins['body_mass_g'])
print(pd.Series(species_model.coef_, index=X_species.columns).round(1))
print('R^2:', round(r2_by_class(penguins['species'], penguins['body_mass_g']), 3))

**Step 4.** Run the permutation test: shuffle the species labels 2,000 times and compute the p-value.

In [ ]:
y_peng = penguins['body_mass_g']
r2_peng = r2_by_class(penguins['species'], y_peng)
shuffled_peng = np.array([r2_by_class(pd.Series(rng.permutation(penguins['species'].to_numpy()), index=penguins.index), y_peng)
                          for _ in range(2000)])
print('observed R^2:', round(r2_peng, 3), '  largest shuffled R^2:', round(shuffled_peng.max(), 3))
print('p-value:', np.mean(shuffled_peng >= r2_peng))

**Step 5.** Bootstrap a 95% confidence interval for each species' mean. Which species differ?

In [ ]:
species_order = ['Adelie', 'Chinstrap', 'Gentoo']
boot_peng = np.array([penguins.sample(len(penguins), replace=True, random_state=rng)
                      .groupby('species')['body_mass_g'].mean()[species_order].to_numpy()
                      for _ in range(2000)])
lo, hi = np.percentile(boot_peng, [2.5, 97.5], axis=0)
pd.DataFrame({'mean': penguins.groupby('species')['body_mass_g'].mean()[species_order],
              'CI low': lo, 'CI high': hi}).round(0)

*Solution:* the p-value is less than 1/2,000, so we reject H₀: the species' population mean body masses are not all equal. The intervals show **Gentoo** penguins are much heavier (about 5,100 g); the Adelie and Chinstrap intervals overlap heavily (both about 3,700 g), so this sample gives no evidence that those two differ.